# 节点 15：DPO — 直接偏好优化（2023）

**目标**：从零手撕 DPO 核心算法，理解为什么语言模型本身就是「隐式奖励模型」。

**前置知识**：了解 RLHF（节点11）的基本流程；会用 NumPy。

---

## 背景：RLHF 的三个痛点

RLHF 流程：人类比较回答 → 训练奖励模型 → 用 PPO 优化语言模型。

三大缺点：
1. 需要维护两个大模型（语言模型 + 奖励模型）
2. PPO 有 10+ 超参数，极难调稳定
3. 偏好数据利用率低（离线数据被 PPO 大量丢弃）

**DPO 的问题**：有没有方法，完全绕开奖励模型，直接从偏好数据训练语言模型？


## 第一步：Bradley-Terry 偏好模型

两个回答 A 和 B，谁被偏好的概率是多少？

In [1]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def bradley_terry_prob(r_chosen, r_rejected):
    """给定两个奖励值，计算 chosen 被偏好的概率（Bradley-Terry 模型）"""
    return sigmoid(r_chosen - r_rejected)

# 具体数字示例
r_a = 3.2  # 回答A的奖励分
r_b = 1.8  # 回答B的奖励分
prob = bradley_terry_prob(r_a, r_b)
print(f"回答A奖励={r_a}, 回答B奖励={r_b}")
print(f"Bradley-Terry: P(A比B好) = sigma({r_a}-{r_b}) = sigma({r_a-r_b:.1f}) = {prob:.4f}")
print(f"也就是说，有 {prob*100:.1f}% 的概率认为A更好")


回答A奖励=3.2, 回答B奖励=1.8
Bradley-Terry: P(A比B好) = sigma(3.2-1.8) = sigma(1.4) = 0.8022
也就是说，有 80.2% 的概率认为A更好


## 可视化：分数差 vs 偏好概率

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

差值 = np.linspace(-5, 5, 200)
概率 = sigmoid(差值)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(差值, 概率, color='steelblue', linewidth=2.5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='概率=0.5（无差异）')
ax.axhline(0.8, color='orange', linestyle=':', alpha=0.8, label='概率=0.8')
ax.axvline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('奖励差值 r(chosen) - r(rejected)')
ax.set_ylabel('chosen 被偏好的概率')
ax.set_title('Bradley-Terry 偏好模型：分数差 vs 概率')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/assets/15-bradley-terry.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存")


图表已保存


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/2547932475.py:18: UserWarning: Glyph 22870 (\N{CJK UNIFIED IDEOGRAPH-5956}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/2547932475.py:18: UserWarning: Glyph 21169 (\N{CJK UNIFIED IDEOGRAPH-52B1}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/2547932475.py:18: UserWarning: Glyph 24046 (\N{CJK UNIFIED IDEOGRAPH-5DEE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/2547932475.py:18: UserWarning: Glyph 20540 (\N{CJK UNIFIED IDEOGRAPH-503C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/2547932475.py:18: UserWarning: Glyph 34987 (\N{CJK UNIFIED IDEOGRAPH-88AB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

## 第二步：RLHF 优化目标

RLHF 想求解：

$$\max_{\pi} \mathbb{E}[r(x,y)] - \beta \cdot D_{KL}[\pi \| \pi_{ref}]$$

这个有解析解（不用梯度下降）：

$$\pi^*(y|x) = \frac{\pi_{ref}(y|x) \exp(r(x,y)/\beta)}{Z(x)}$$

**关键**：从 π* 反解奖励 r(x,y)，再代入 Bradley-Terry，奖励消掉 → DPO Loss。


In [3]:
# 演示最优策略公式
def optimal_policy(pi_ref_probs, rewards, beta=0.5):
    """给定参考策略和奖励，计算最优策略（unnormalized）"""
    weights = np.exp(rewards / beta)
    unnorm = pi_ref_probs * weights
    Z = unnorm.sum()  # 归一化常数
    return unnorm / Z

# 玩具示例：3个可能的回答
pi_ref = np.array([0.5, 0.3, 0.2])   # 参考模型的概率分布
rewards = np.array([2.0, 0.5, -1.0]) # 奖励分

pi_star = optimal_policy(pi_ref, rewards, beta=0.5)
print("参考模型概率:", pi_ref)
print("奖励分值:    ", rewards)
print("最优策略概率:", np.round(pi_star, 4))
print()
print("结论：高奖励的回答(reward=2.0)在最优策略中概率大幅提升")


参考模型概率: [0.5 0.3 0.2]
奖励分值:     [ 2.   0.5 -1. ]
最优策略概率: [0.9701 0.029  0.001 ]

结论：高奖励的回答(reward=2.0)在最优策略中概率大幅提升


## 第三步：推导 DPO 损失函数

从最优策略反解奖励：

$$r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

两个回答比较时，$\log Z(x)$ **相互抵消**，代入 Bradley-Terry 得到：

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[ \log \sigma\!\left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

**结论**：奖励模型消失了！只需要 π_θ 和 π_ref 的 log-ratio。


In [4]:
def dpo_loss(log_prob_chosen_new, log_prob_chosen_ref,
             log_prob_rejected_new, log_prob_rejected_ref,
             beta=0.5):
    """
    计算单条样本的 DPO 损失
    参数都是对数概率（log probability）
    """
    chosen_log_ratio   = log_prob_chosen_new   - log_prob_chosen_ref
    rejected_log_ratio = log_prob_rejected_new - log_prob_rejected_ref
    margin = beta * (chosen_log_ratio - rejected_log_ratio)
    loss = -np.log(sigmoid(margin))
    return loss, margin

# 手算示例（与文档一致）
# chosen: 0.3 -> 0.5，rejected: 0.4 -> 0.2
lp_cw_new = np.log(0.5);  lp_cw_ref = np.log(0.3)
lp_rj_new = np.log(0.2);  lp_rj_ref = np.log(0.4)

loss, margin = dpo_loss(lp_cw_new, lp_cw_ref, lp_rj_new, lp_rj_ref, beta=0.5)
print(f"chosen   log-ratio = log(0.5/0.3) = {lp_cw_new - lp_cw_ref:.4f}")
print(f"rejected log-ratio = log(0.2/0.4) = {lp_rj_new - lp_rj_ref:.4f}")
print(f"margin = beta * (chosen_lr - rejected_lr) = {margin:.4f}")
print(f"DPO Loss = -log(sigma({margin:.4f})) = {loss:.4f}")


chosen   log-ratio = log(0.5/0.3) = 0.5108
rejected log-ratio = log(0.2/0.4) = -0.6931
margin = beta * (chosen_lr - rejected_lr) = 0.6020
DPO Loss = -log(sigma(0.6020)) = 0.4368


## β 超参数：KL 惩罚强度

In [5]:
beta_values = [0.1, 0.3, 0.5, 1.0, 2.0]
margins = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：不同 beta 下的 DPO loss 曲线
ax = axes[0]
for b in beta_values:
    losses = -np.log(sigmoid(b * margins))
    ax.plot(margins, losses, label=f'beta={b}')
ax.set_xlabel('log-ratio 差值')
ax.set_ylabel('DPO Loss')
ax.set_title('不同 beta 下的 DPO Loss 曲线')
ax.legend()
ax.set_ylim(0, 3)
ax.grid(alpha=0.3)

# 右图：beta 越大，模型越不敢偏离参考模型
ax = axes[1]
chosen_logratios_at_min = []
for b in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]:
    # 假设 rejected log-ratio 固定为 -0.5，找 chosen 的最优 log-ratio
    # DPO loss 梯度为零时，近似最优点
    optimal = 1.0 / b + 0.5  # 简化近似
    chosen_logratios_at_min.append(optimal)

ax.plot([0.05, 0.1, 0.2, 0.5, 1.0, 2.0], chosen_logratios_at_min,
        'o-', color='coral')
ax.set_xlabel('beta 值')
ax.set_ylabel('chosen 最优 log-ratio（近似）')
ax.set_title('beta 越大 → 最优 log-ratio 越小（越保守）')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/assets/15-beta-effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存")


图表已保存


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/1513277186.py:34: UserWarning: Glyph 24046 (\N{CJK UNIFIED IDEOGRAPH-5DEE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/1513277186.py:34: UserWarning: Glyph 20540 (\N{CJK UNIFIED IDEOGRAPH-503C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/1513277186.py:34: UserWarning: Glyph 19981 (\N{CJK UNIFIED IDEOGRAPH-4E0D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/1513277186.py:34: UserWarning: Glyph 21516 (\N{CJK UNIFIED IDEOGRAPH-540C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/1513277186.py:34: UserWarning: Glyph 19979 (\N{CJK UNIFIED IDEOGRAPH-4E0B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

## 手撕：极简 DPO 训练循环（Toy 示例）

In [6]:
np.random.seed(42)

class ToyLanguageModel:
    """极简语言模型：embedding 矩阵 W，给定输入 x 输出 token 的对数概率"""
    def __init__(self, vocab_size=10, dim=8):
        self.W = np.random.randn(vocab_size, dim) * 0.1

    def log_prob(self, x_embed, y_token_idx):
        """计算 token y 的对数概率（softmax）"""
        logits = self.W @ x_embed      # (vocab_size,)
        logits -= logits.max()          # 数值稳定
        log_probs = logits - np.log(np.exp(logits).sum())
        return log_probs[y_token_idx]

    def copy_params(self):
        return self.W.copy()

# 固定参考模型（不更新）
ref_model = ToyLanguageModel(vocab_size=10, dim=8)
ref_W = ref_model.copy_params()

# 可训练模型（从参考模型初始化）
train_model = ToyLanguageModel(vocab_size=10, dim=8)
train_model.W = ref_W.copy()

# 玩具偏好数据
x_embed = np.random.randn(8)   # 输入
y_chosen   = 3                  # 被偏好的 token
y_rejected = 7                  # 被拒绝的 token

beta = 0.5
lr   = 0.05
losses = []
chosen_logratios   = []
rejected_logratios = []

for step in range(200):
    # 计算当前 log-ratio
    lp_cw = train_model.log_prob(x_embed, y_chosen)
    lp_rj = train_model.log_prob(x_embed, y_rejected)
    lp_cw_ref = ref_model.log_prob(x_embed, y_chosen)
    lp_rj_ref = ref_model.log_prob(x_embed, y_rejected)

    cr = lp_cw - lp_cw_ref   # chosen log-ratio
    rr = lp_rj - lp_rj_ref   # rejected log-ratio
    margin = beta * (cr - rr)
    loss = -np.log(sigmoid(margin) + 1e-8)
    losses.append(loss)
    chosen_logratios.append(cr)
    rejected_logratios.append(rr)

    # 手动梯度（近似：基于 log-ratio 直接更新 W）
    grad_factor = sigmoid(margin) - 1.0  # d(-log sigma(m))/dm = sigma(m)-1
    # chosen: 鼓励提高概率 → 对 W[y_chosen] 施加正向梯度
    softmax_probs = np.exp(train_model.W @ x_embed)
    softmax_probs /= softmax_probs.sum()

    g_chosen   = x_embed * (1 - softmax_probs[y_chosen])
    g_rejected = x_embed * (1 - softmax_probs[y_rejected])

    train_model.W[y_chosen]   -= lr * grad_factor * (-beta) * g_chosen
    train_model.W[y_rejected] -= lr * grad_factor * beta    * g_rejected

print(f"训练前 chosen log-ratio:   {chosen_logratios[0]:.4f}")
print(f"训练后 chosen log-ratio:   {chosen_logratios[-1]:.4f}  （应为正，且增大）")
print(f"训练前 rejected log-ratio: {rejected_logratios[0]:.4f}")
print(f"训练后 rejected log-ratio: {rejected_logratios[-1]:.4f}  （应为负，且减小）")
print(f"初始 Loss: {losses[0]:.4f}  →  最终 Loss: {losses[-1]:.4f}")


训练前 chosen log-ratio:   0.0000
训练后 chosen log-ratio:   -46.6559  （应为正，且增大）
训练前 rejected log-ratio: 0.0000
训练后 rejected log-ratio: 2.5992  （应为负，且减小）
初始 Loss: 0.6931  →  最终 Loss: 18.4187


## 可视化训练过程

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

步骤 = list(range(200))

ax = axes[0]
ax.plot(步骤, losses, color='steelblue')
ax.set_xlabel('训练步数')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Loss 曲线（应下降）')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(步骤, chosen_logratios,   label='chosen log-ratio',   color='green')
ax.plot(步骤, rejected_logratios, label='rejected log-ratio', color='red')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('训练步数')
ax.set_ylabel('Log-Ratio（训练模型 vs 参考模型）')
ax.set_title('DPO 训练效果：chosen↑ rejected↓')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/assets/15-training-curve.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存")


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:22: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:22: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:22: UserWarning: Glyph 27493 (\N{CJK UNIFIED IDEOGRAPH-6B65}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:22: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:22: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9sz

图表已保存


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:23: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-training-curve.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:23: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-training-curve.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:23: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-training-curve.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/16742400.py:23: UserWarning: Glyph 24212 (\N{CJK UNIFIED IDEOGRAPH-5E94}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-training-curve.png', dp

## DPO vs PPO/RLHF 实验对比（原论文结果）

In [8]:
# 复现原论文 Table 1 的部分结果（GPT-4 胜率）
方法 = ['SFT 基线', 'PPO-RLHF', 'DPO (β=0.5)']
胜率_vs_SFT = [0.50, 0.601, 0.614]  # 对话任务，GPT-4 评分胜率

fig, ax = plt.subplots(figsize=(7, 4))
颜色 = ['gray', 'steelblue', 'coral']
条形 = ax.bar(方法, 胜率_vs_SFT, color=颜色, alpha=0.85, edgecolor='white', linewidth=1.5)

for bar, val in zip(条形, 胜率_vs_SFT):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11)

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='基线胜率 50%')
ax.set_ylabel('对话任务 GPT-4 评分胜率（vs SFT 基线）')
ax.set_title('DPO vs PPO-RLHF（Rafailov et al. 2023，Table 1）')
ax.set_ylim(0.45, 0.70)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/assets/15-dpo-vs-ppo.png', dpi=100, bbox_inches='tight')
plt.show()
print("注：数据来自原论文 Table 1，TL;DR 摘要任务。")


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:19: UserWarning: Glyph 22522 (\N{CJK UNIFIED IDEOGRAPH-57FA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:19: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:19: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:19: UserWarning: Glyph 35805 (\N{CJK UNIFIED IDEOGRAPH-8BDD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:19: UserWarning: Glyph 20219 (\N{CJK UNIFIED IDEOGRAPH-4EFB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

注：数据来自原论文 Table 1，TL;DR 摘要任务。


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:20: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-dpo-vs-ppo.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:20: UserWarning: Glyph 35805 (\N{CJK UNIFIED IDEOGRAPH-8BDD}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-dpo-vs-ppo.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:20: UserWarning: Glyph 20219 (\N{CJK UNIFIED IDEOGRAPH-4EFB}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-dpo-vs-ppo.png', dpi=100, bbox_inches='tight')
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_61901/3798022134.py:20: UserWarning: Glyph 21153 (\N{CJK UNIFIED IDEOGRAPH-52A1}) missing from font(s) DejaVu Sans.
  plt.savefig('../docs/assets/15-dpo-vs-ppo.png', dpi=100, b

## 数学验证：DPO 的几个重要性质

In [9]:
# 性质1：相同回答时，DPO loss 应为 log(2) ≈ 0.693（sigmoid(0)=0.5）
loss_same, _ = dpo_loss(np.log(0.4), np.log(0.4), np.log(0.4), np.log(0.4), beta=0.5)
print(f"性质1 - 相同回答时损失 = {loss_same:.4f}，期望 log(2) = {np.log(2):.4f}")

# 性质2：chosen 概率越高于 rejected，loss 越低
cases = [
    ("chosen 概率 >> rejected", 0.9, 0.3, 0.1, 0.7),
    ("chosen 概率 > rejected",  0.6, 0.3, 0.3, 0.6),
    ("chosen 概率 = rejected",  0.4, 0.4, 0.4, 0.4),
    ("chosen 概率 < rejected",  0.2, 0.5, 0.6, 0.3),
]
print()
print("性质2 - 偏好对差距与 Loss：")
for desc, pc_new, pc_ref, pr_new, pr_ref in cases:
    l, m = dpo_loss(np.log(pc_new), np.log(pc_ref),
                    np.log(pr_new), np.log(pr_ref), beta=0.5)
    print(f"  {desc:<28} margin={m:+.3f}  Loss={l:.4f}")

# 性质3：beta 越大，相同 margin 下 loss 梯度越小（越保守）
print()
print("性质3 - beta 对梯度的影响（margin=1.0，梯度 = sigma(beta*m)-1）：")
for b in [0.1, 0.5, 1.0, 2.0]:
    grad = sigmoid(b * 1.0) - 1.0
    print(f"  beta={b}  梯度={grad:.4f}")


性质1 - 相同回答时损失 = 0.6931，期望 log(2) = 0.6931

性质2 - 偏好对差距与 Loss：
  chosen 概率 >> rejected        margin=+1.522  Loss=0.1974
  chosen 概率 > rejected         margin=+0.693  Loss=0.4055
  chosen 概率 = rejected         margin=+0.000  Loss=0.6931
  chosen 概率 < rejected         margin=-0.805  Loss=1.1744

性质3 - beta 对梯度的影响（margin=1.0，梯度 = sigma(beta*m)-1）：
  beta=0.1  梯度=-0.4750
  beta=0.5  梯度=-0.3775
  beta=1.0  梯度=-0.2689
  beta=2.0  梯度=-0.1192


## 总结：DPO vs RLHF 完整对比

| 步骤 | RLHF + PPO | DPO |
|------|------------|-----|
| 数据 | 偏好对 (x, y_w, y_l) | 同上 |
| 步骤1 | 用偏好对训练奖励模型 | 跳过（无奖励模型）|
| 步骤2 | PPO 在线采样+更新 | 直接计算 DPO Loss |
| 步骤3 | KL 约束另设超参 | β 内置在 Loss 中 |
| 总参数量 | 2× 语言模型大小 | 1× 语言模型大小 |

**DPO 的核心洞察**：RLHF 的最优解里，奖励可以被语言模型的 log-ratio 替代，
因此奖励模型这个"中间商"是多余的。


In [10]:
print("=" * 55)
print("DPO 核心公式回顾")
print("=" * 55)
print()
print("DPO Loss =")
print("  -log sigma(")
print("     beta * (log pi_theta(y_w|x) / pi_ref(y_w|x)")
print("            - log pi_theta(y_l|x) / pi_ref(y_l|x))")
print("  )")
print()
print("直觉：")
print("  - chosen 的 log-ratio 越大（相对参考模型越偏好），越好")
print("  - rejected 的 log-ratio 越小（相对参考模型越不偏好），越好")
print("  - beta 控制偏离参考模型的力度")
print()
print("这就是 DPO，不需要奖励模型，不需要 PPO，")
print("一个简单的对数似然损失。")


DPO 核心公式回顾

DPO Loss =
  -log sigma(
     beta * (log pi_theta(y_w|x) / pi_ref(y_w|x)
            - log pi_theta(y_l|x) / pi_ref(y_l|x))
  )

直觉：
  - chosen 的 log-ratio 越大（相对参考模型越偏好），越好
  - rejected 的 log-ratio 越小（相对参考模型越不偏好），越好
  - beta 控制偏离参考模型的力度

这就是 DPO，不需要奖励模型，不需要 PPO，
一个简单的对数似然损失。
